# Processing: Station Nord 2024

In this notebook, we'll run thermal and solar simulations using radiosonde and ceilometer data from Station Nord, Greenland (2024 campaign).

```{note}
**Campaign data required.** This notebook uses radiosonde and ceilometer data from Station Nord that is not publicly available. You can adapt the workflow to your own data by replacing the file paths.
```

## Setup

In [ ]:
ds_rs = xr.open_mfdataset(glob.glob('/projekt_agmwend/home_rad/Joshua/STN2024/RS_STN/radiosonde_netcdf/*.nc'), combine='by_coords')

In [ ]:
ds_rs

In [ ]:
import glob
ds_ceilo = xr.open_dataset('/projekt_agmwend/data/Station_Nord_2024/Ceilometer/Ceilometer_STN_all.nc')

In [ ]:
import xarray as xr
from datetime import datetime
from pyradtran.io_unified import RadiosondeAtmosphereGenerator
from pathlib import Path


# 2. Build a LibRadtran atmosphere file from the closest IGRA sounding
atm_path = RadiosondeAtmosphereGenerator.create_radiosonde_atmosphere_file(
    time=datetime(2023, 12, 30, 0),
    latitude=lat,
    longitude=lon,
    output_filepath="work/sonde_nya.dat",
)

In [ ]:
ds_gp = ds_rs[['altitude', 'air_temperature', 'relative_humidity']].groupby('time.dayofyear')
atm_paths = []
for day, ds_day in ds_gp:
    print(f"Day {day}:")
    print(ds_day)
    date = ds_day['time'].dt.date.values[0]
    df_day = ds_day.to_dataframe().reset_index().dropna(how='any').sort_values(by='altitude', ascending=True).drop(columns=['time']).drop_duplicates()
    df_day['altitude'] /= 1000
    df_day['air_temperature'] += 273.15
    atm_name = f"work/stn2024_day_{date}.csv"
    df_day.to_csv(atm_name, index=False, header=False, sep='\t')
    atm_paths.append(atm_name)


In [ ]:
atm_paths[11]

In [ ]:
ds_sim_thermal = ds.isel(time=slice(0,2)).pyradtran.run(
    config_path=thermal_config_path,
    return_dataset=True,
    save_to_file=False,
    params={"atmosphere_file": str(atm_paths[10])},
    show_progress=False,
)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 4), sharex=True)

(ds_sim_solar.drop_vars(['latitude', 'longitude', 'altitude']).eglo / 1000).plot()
(ds_sim_thermal.drop_vars(['latitude', 'longitude', 'altitude']).eglo).plot(ax=ax[0])

for a in ax:
    a.set_ylabel('F_down (W/m²)')
    a.grid()
    a.spines[['top', 'right']].set_visible(False)

In [ ]:


# 3. Run pyradtran from the xarray accessor, overriding the atmosphere file
result_ds = ds.pyradtran.run(
    config_path=solar_config_path,
    params={"radiosonde": str(atm_path)+" H2O RH"},
    return_dataset=True,
    save_to_file=False,
    show_progress=False,
)
